# 03 · 让一组软画笔自己学着画图

想象画布上有 24 支喷笔。每支都能移动、变宽、换颜色、调浓淡。我们给它们一张目标图，让它们边画边比较，逐步找到更合适的位置和颜色。

这一篇把“手动设参数画图”变成“根据像素误差自动调参数”。实验在二维画布上进行，使用圆形高斯和加法混合；通过它，可以清楚地看见**渲染 → 误差 → 梯度 → 更新**这条链。

在本主题环境从第一格运行。默认画布是 64×48、24 支软画笔、300 次更新，使用 CPU。

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(42)
height, width = 48, 64
num_gaussians = 24
num_steps = 300

先保持这张小画布。渲染时，每个像素都要计算每支画笔的贡献，主要中间张量包含 H×W×N 项；提高分辨率或画笔数量，计算量也会随之增加。

## 1. 先摆好要临摹的画

目标由红、绿、蓝三团平滑光斑组成：位置不同，宽度也不同。它足够简单，让你能看清哪里应该添色、哪里应该变淡。

代码用公式生成目标，随后把它作为一张普通像素图交给模型。

In [ ]:
yy, xx = torch.meshgrid(torch.arange(height).float(), torch.arange(width).float(), indexing="ij")
red = torch.exp(-((xx - 18) ** 2 + (yy - 16) ** 2) / (2 * 6 ** 2))
green = torch.exp(-((xx - 43) ** 2 + (yy - 18) ** 2) / (2 * 7 ** 2))
blue = torch.exp(-((xx - 32) ** 2 + (yy - 34) ** 2) / (2 * 5 ** 2))
target = torch.stack([red, green, blue], dim=-1)
plt.figure(figsize=(6, 4))
plt.imshow(target.numpy())
plt.title("Synthetic target")
plt.show()

把这张图当作摆在旁边的参考画。接下来随机放置的 24 支画笔要合作，把各自的贡献叠成尽量接近它的图像。每张图都用 `(H,W,3)` 存储，最后三项对应 RGB。

## 2. 给每支画笔装上四组旋钮

- **位置**：落笔点放在哪里？
- **半径**：喷出的光斑铺多开？
- **颜色**：红、绿、蓝分别有多少？
- **不透明度**：这一笔有多浓？

这些旋钮都由可学习参数控制。颜色和不透明度经过 sigmoid，落在 0 到 1；两个非负尺度取平均，得到圆形足迹的有效半径。

一次渲染，就是计算每支笔在每个像素上留下多少颜色，然后把贡献相加。

### 实现：从参数表生成整张画布

`MultiGaussianModel` 保存所有画笔的参数，`render_multi_gaussians` 生成像素网格，再计算网格与各个中心之间的距离。

广播让所有像素、所有画笔一起计算。这里的参数、网格和常量都在 CPU；以后搬到 GPU 时，把它们一起移动即可。

In [ ]:
class MultiGaussianModel(nn.Module):
    def __init__(self, num_gaussians, target_h, target_w):
        super().__init__()
        self.num_gaussians = num_gaussians

        # 1. 位置: 随机分布在图像范围内
        # 范围: [-W/2, W/2] 和 [-H/2, H/2]
        self.pos = nn.Parameter(torch.rand(num_gaussians, 2) * torch.tensor([target_w, target_h]) - torch.tensor([target_w/2, target_h/2]))

        # 2. 颜色: 随机 RGB
        self.color = nn.Parameter(torch.rand(num_gaussians, 3))

        # 3. 缩放: 初始较小，避免一开始就糊成一团
        self.scale = nn.Parameter(torch.rand(num_gaussians, 2) * 5.0 + 1.0)

        # 4. 不透明度: 初始较低
        self.opacity_logit = nn.Parameter(torch.ones(num_gaussians) * -2.0) # sigmoid(-2) ≈ 0.12

    def forward(self):
        return {
            'pos': self.pos,
            'color': torch.sigmoid(self.color),
            'scale': torch.abs(self.scale),
            'opacity': torch.sigmoid(self.opacity_logit)
        }


def render_multi_gaussians(params, H, W):
    """
    并行渲染多个 2D 高斯
    """
    num_gaussians = params['pos'].shape[0]

    # 1. 生成像素网格 (H, W, 2)
    y_coords, x_coords = torch.meshgrid(
        torch.arange(H).float(), 
        torch.arange(W).float(),
        indexing='ij'
    )
    # grid: (H, W, 2)
    grid = torch.stack([x_coords, y_coords], dim=-1)

    # 2. 调整中心坐标
    # params['pos']: (N, 2) -> center: (N, 2)
    centers = params['pos'] + torch.tensor([W/2, H/2])

    # 3. 计算距离 (广播机制)
    # grid: (H, W, 1, 2)
    # centers: (1, 1, N, 2)
    # delta: (H, W, N, 2)
    delta = grid.unsqueeze(2) - centers.reshape(1, 1, num_gaussians, 2)
    dist_sq = torch.sum(delta**2, dim=-1) # (H, W, N)

    # 4. 计算高斯强度
    # scale: (N, 2) -> mean_scale: (N,)
    # radius: (1, 1, N)
    radius = params['scale'].mean(dim=1).reshape(1, 1, num_gaussians)

    # power: (H, W, N)
    power = torch.exp(-0.5 * dist_sq / (radius**2 + 1e-6))

    # 5. 混合 (简化版：加法混合，不考虑遮挡)
    # 真正的 3DGS 需要排序和 Alpha Blending，但在 2D 拟合中，加法混合也能出效果，且更快
    # alpha: (H, W, N)
    alpha = (params['opacity'].reshape(1, 1, num_gaussians) * power)

    # contribution: (H, W, N, 3)
    # color: (1, 1, N, 3)
    contribution = alpha.unsqueeze(-1) * params['color'].reshape(1, 1, num_gaussians, 3)

    # sum over gaussians: (H, W, 3)
    final_image = torch.sum(contribution, dim=2)

    # 限制在 0-1
    return torch.clamp(final_image, 0.0, 1.0)

In [ ]:
model = MultiGaussianModel(num_gaussians, height, width)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
with torch.no_grad():
    initial_image = render_multi_gaussians(model(), height, width).clone()
params = model()
for name, value in params.items():
    print(name, tuple(value.shape))
plt.figure(figsize=(6, 4))
plt.imshow(initial_image.numpy())
plt.title("Before fitting")
plt.show()

这张初始图相当于画笔们随手落下的第一稿。代码把它保存为 `initial_image`，训练结束后再拿出来对照，看看位置、大小和颜色怎样共同改变画面。

## 3. 像素误差怎样告诉画笔往哪调？

对每个像素比较预测与目标，平方后求平均，得到 MSE：

$$L=\operatorname{mean}\bigl((I_{render}-I_{target})^2\bigr).$$

反向传播沿着“颜色贡献 → 高斯权重 → 距离与参数”的链条，计算各个旋钮改变一点会怎样影响误差。Adam 再利用这些梯度更新参数。

这一轮做完，重新画一张，再比较一次。整段渲染和损失都留在 PyTorch 里，自动微分就能沿计算链把反馈传回画笔。

<details>
<summary>调参数时，哪些地方值得留意？</summary>

很淡、远离目标的足迹可能收到较弱梯度；输出 `clamp` 会截断超出显示范围的值，饱和区的梯度也会受影响。遇到训练停滞时，可回看初始化、opacity 和画笔半径。

本篇固定画笔数量。完整 3DGS 还会通过克隆、分裂和修剪调整高斯集合，并把相机与多视角信息接入渲染。

</details>

In [ ]:
loss_history = []
for step in range(num_steps):
    prediction = render_multi_gaussians(model(), height, width)
    loss = (prediction - target).square().mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())
    if (step + 1) % 50 == 0:
        print(f"Step {step + 1}/{num_steps}, MSE={loss.item():.6f}")

每次打印的 MSE 都来自当前一整张画。画笔的位置、半径、颜色和浓淡一起调整，误差通过同一条计算链反馈给它们。

## 4. 把参考画、第一稿和最后一稿放在一起

最后一次更新后再渲染一遍，得到最终图。下面并排展示目标、初始图、拟合图和平均绝对误差。

先看光斑位置，再看宽度和颜色。误差图则像一张批注：越亮的地方，像素差异越大。

In [ ]:
with torch.no_grad():
    final_image = render_multi_gaussians(model(), height, width)
    error = (final_image - target).abs().mean(dim=-1)
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, image, title in zip(
    axes, [target.numpy(), initial_image.numpy(), final_image.numpy(), error.numpy()],
    ["Target", "Initial", "Fitted", "Mean absolute error"],
):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()
plt.figure(figsize=(7, 3))
plt.plot(loss_history)
plt.xlabel("Update step")
plt.ylabel("Pixel MSE")
plt.title("Gaussian image fitting")
plt.show()

同一片亮斑，可以由一支较宽的画笔画出，也可以由几支较窄的笔叠出来。这个实验学习的是一组能描出目标图的二维参数。

继续走向三维重建时，就要让同一组空间高斯在不同相机视角下都能解释照片；多视角约束会进一步限制它们的几何位置。

## 选读 A · 只留一支笔，会发生什么？

先把画笔数量降到一支，更容易看清一个参数怎样影响整个光斑。下面保留单高斯版本，真正进入图像计算的是 XY 位置、尺度均值、颜色和 opacity。

<details>
<summary>沿着代码看参数有没有用上</summary>

`GaussianModel` 还保存了 Z 和旋转，但这个圆形渲染器没有使用它们，所以像素损失不会更新这些几何量。`compute_covariance_3d_torch` 演示了 Z 轴旋转的协方差构造，当前绘图函数也没有调用它。

可以从 `render_single_gaussian_torch` 的输入一路追到输出，看看哪些参数真正连在计算链上。

</details>

In [ ]:
class GaussianModel(nn.Module):
    def __init__(self):
        super().__init__()
        # 初始化一个高斯球
        # 我们希望它能自动移动到目标位置，并变成目标颜色

        # 1. 位置: 随机初始化
        self.position = nn.Parameter(torch.tensor([0.0, 0.0, 5.0]))

        # 2. 颜色: 随机初始化 (RGB)
        self.color = nn.Parameter(torch.rand(3))

        # 3. 缩放: 初始为一个单位球
        self.scale = nn.Parameter(torch.tensor([1.0, 1.0, 1.0]))

        # 4. 旋转: 初始为不旋转 (欧拉角)
        self.rotation = nn.Parameter(torch.zeros(3))

        # 5. 不透明度: 初始为 0.5 (Logit 空间，经过 Sigmoid 后变成 0.5)
        self.opacity_logit = nn.Parameter(torch.tensor([0.0]))

    def forward(self):
        # 返回处理后的参数 (例如应用激活函数)
        return {
            'pos': self.position,
            'color': torch.sigmoid(self.color), # 限制在 0-1
            'scale': torch.abs(self.scale), # 必须为正
            'rot': self.rotation,
            'opacity': torch.sigmoid(self.opacity_logit) # 限制在 0-1
        }


def compute_covariance_3d_torch(scale, rotation):
    # 构建缩放矩阵 S
    S = torch.diag(scale)

    # 构建旋转矩阵 R (欧拉角 -> 矩阵)
    x, y, z = rotation[0], rotation[1], rotation[2]

    # 注意：这里需要手动构建 PyTorch 的旋转矩阵
    # 为了简化，我们先只写 Z 轴旋转
    # 实际应该写完整的 XYZ 旋转
    cos_z = torch.cos(z)
    sin_z = torch.sin(z)
    R_z = torch.stack([
        torch.stack([cos_z, -sin_z, torch.tensor(0.0)]),
        torch.stack([sin_z, cos_z, torch.tensor(0.0)]),
        torch.stack([torch.tensor(0.0), torch.tensor(0.0), torch.tensor(1.0)])
    ])

    R = R_z # 简化版

    # Sigma = R @ S @ S.T @ R.T
    Sigma = R @ S @ S.t() @ R.t()
    return Sigma


def render_single_gaussian_torch(params, target_h, target_w):
    """
    极简版的可微渲染器：只渲染一个高斯球
    """
    # 1. 准备画布 (H, W, 3)
    # 注意：这里使用网格上的连续表达式构建图像
    # 可微性取决于计算链，而不是是否预先创建了零张量

    y_coords, x_coords = torch.meshgrid(
        torch.arange(target_h).float(), 
        torch.arange(target_w).float(),
        indexing='ij'
    )

    # 2. 投影 (简化版：假设正交投影，直接忽略 Z，只看 XY)
    # 这是一个巨大的简化，为了先跑通优化循环
    # 实际上应该用 EWA Splatting

    center_x = params['pos'][0] + target_w / 2
    center_y = params['pos'][1] + target_h / 2

    # 3. 计算每个像素到中心的距离平方
    dx = x_coords - center_x
    dy = y_coords - center_y
    dist_sq = dx**2 + dy**2

    # 4. 高斯衰减
    # radius = scale.mean() * 10 (随便拍的一个系数)
    radius = params['scale'].mean() * 20.0
    power = torch.exp(-0.5 * dist_sq / (radius**2 + 1e-6))

    # 5. 最终图像
    # Image = Color * Opacity * Power
    # 扩展维度以便广播: (H, W, 1) * (3,)
    pixel_alpha = (params['opacity'] * power).unsqueeze(-1)
    pixel_color = params['color'] * pixel_alpha

    return pixel_color

### 用柔边画笔临摹一个硬边圆盘

目标是一块绿色圆盘，画笔则从中心向外平滑变淡。观察最终图时，尤其看圆盘边缘：一支柔边画笔会怎样折中半径和浓淡？

把下一格 `RUN_SINGLE_GAUSSIAN` 改成 `True`，就会进行 100 次更新，并排显示目标、结果与损失曲线。

In [ ]:
RUN_SINGLE_GAUSSIAN = False
if RUN_SINGLE_GAUSSIAN:
    single_h = single_w = 64
    sy, sx = torch.meshgrid(torch.arange(single_h), torch.arange(single_w), indexing="ij")
    single_target = torch.zeros(single_h, single_w, 3)
    single_target[..., 1] = ((sx - 40).square() + (sy - 20).square() < 100).float()
    single_model = GaussianModel()
    single_optimizer = torch.optim.Adam(single_model.parameters(), lr=0.1)
    single_losses = []
    for _ in range(100):
        image = render_single_gaussian_torch(single_model(), single_h, single_w)
        single_loss = (image - single_target).square().mean()
        single_optimizer.zero_grad()
        single_loss.backward()
        single_optimizer.step()
        single_losses.append(single_loss.item())
    with torch.no_grad():
        single_final = render_single_gaussian_torch(single_model(), single_h, single_w)
    fig, axes = plt.subplots(1, 3, figsize=(11, 3))
    axes[0].imshow(single_target.numpy()); axes[0].set_title("Target")
    axes[1].imshow(single_final.numpy()); axes[1].set_title("One soft brush")
    axes[2].plot(single_losses); axes[2].set_title("Pixel MSE")
    plt.tight_layout()
    plt.show()

## 选读 B · 换成自己的小图

把 `FIT_LOCAL_IMAGE` 改成 `True`，下面就会读取本目录的 `IP.png`，缩小到 64×48，再用 100 支软画笔临摹。

也可以把路径换成自己的图片。先看大块颜色怎样被拼出来，再看细节需要多少笔触。加法混合会把各笔颜色累加，适合用来观察这种“多团光斑拼图”的过程。

In [ ]:
FIT_LOCAL_IMAGE = False
if FIT_LOCAL_IMAGE:
    import numpy as np
    from PIL import Image
    picture = Image.open("IP.png").convert("RGB").resize((64, 48))
    picture_target = torch.from_numpy(np.asarray(picture).copy()).float() / 255.0
    picture_model = MultiGaussianModel(100, 48, 64)
    picture_optimizer = torch.optim.Adam(picture_model.parameters(), lr=0.1)
    for step in range(300):
        picture_prediction = render_multi_gaussians(picture_model(), 48, 64)
        picture_loss = (picture_prediction - picture_target).square().mean()
        picture_optimizer.zero_grad()
        picture_loss.backward()
        picture_optimizer.step()
    with torch.no_grad():
        picture_final = render_multi_gaussians(picture_model(), 48, 64)
    fig, axes = plt.subplots(1, 2, figsize=(8, 3))
    axes[0].imshow(picture_target.numpy()); axes[0].set_title("Local target")
    axes[1].imshow(picture_final.numpy()); axes[1].set_title("100 soft brushes")
    plt.tight_layout()
    plt.show()

## 试着改一个旋钮

<details>
<summary>画笔为什么能根据图像误差移动？</summary>

像素权重是中心位置的连续函数。中心改变会改变各像素亮度，PyTorch 能沿这条关系计算误差对位置的梯度。

</details>

<details>
<summary>为什么两次训练可能画得很像，参数却不同？</summary>

多支画笔可以用不同的位置、半径、颜色和浓淡组合，画出接近的像素结果。

</details>

<details>
<summary>为什么训练时把损失计算留在 PyTorch 里？</summary>

PyTorch 会记录渲染运算之间的依赖。反向传播沿着这份记录回到参数；转换成普通 NumPy 数组后，这段自动微分记录就断开了。

</details>

把 24 支画笔改成 8 支或 48 支，比较哪些细节先丢失、哪些先变好。还可以打开[交互点云选读](04_交互点云与实现边界.ipynb)，看看离散点着色与连续高斯足迹的不同计算方式。